In [5]:
import pandas as pd
import json
import re
from pathlib import Path
from collections import Counter

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
# Define the folder containing the raw Adzuna JSON files
DATA_DIR = Path("../data/raw/adzuna")

# Raw job-posting files collected for the project
files = [
    "adzuna_software_engineer_2026-09-29.json",
    "adzuna_business_analyst_2026-09-29.json",
    "adzuna_data_analyst_2026-09-29.json",
    "adzuna_digital_marketing_2026-09-29.json"
]

print("Data directory:", DATA_DIR)
print("Files to load:")
for file in files:
    print("-", file)

Data directory: ../data/raw/adzuna
Files to load:
- adzuna_software_engineer_2026-09-29.json
- adzuna_business_analyst_2026-09-29.json
- adzuna_data_analyst_2026-09-29.json
- adzuna_digital_marketing_2026-09-29.json


In [7]:
# Load all job postings from the raw JSON files

all_jobs = []

for file in files:
    file_path = DATA_DIR / file
    
    with open(file_path, "r", encoding="utf-8") as f:
        jobs = json.load(f)
    
    print(f"{file}: {len(jobs)} records")
    all_jobs.extend(jobs)

print("\nTotal job postings loaded:", len(all_jobs))

adzuna_software_engineer_2026-09-29.json: 625 records
adzuna_business_analyst_2026-09-29.json: 625 records
adzuna_data_analyst_2026-09-29.json: 625 records
adzuna_digital_marketing_2026-09-29.json: 625 records

Total job postings loaded: 2500


In [8]:
# Convert the combined job postings into a DataFrame
df = pd.DataFrame(all_jobs)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (2500, 17)

Columns:
['redirect_url', 'description', 'created', 'contract_time', 'adref', 'id', 'location', 'company', 'longitude', 'latitude', '__CLASS__', 'category', 'salary_is_predicted', 'title', 'contract_type', 'salary_max', 'salary_min']


In [9]:
# Inspect the fields relevant to text analysis

print("Description data type:")
print(df["description"].map(type).value_counts())

print("\nMissing values:")
print(df[["id", "title", "description", "category"]].isnull().sum())

print("\nSample job description:")
print(df["description"].iloc[0][:1000])

Description data type:
description
<class 'str'>    2500
Name: count, dtype: int64

Missing values:
id             0
title          0
description    0
category       0
dtype: int64

Sample job description:
This job is with ABB, an inclusive employer and a member of myGwork – the largest global platform for the LGBTQ business community. Please do not contact the recruiter directly. At ABB, we help industries run leaner and cleaner—and every person here makes that happen. You’ll be empowered to lead, supported to grow, and proud of the impact we create together. Join us and help run what runs the world. This Position reports to: Digital Solution Engineering Manager What we believe in ABB’s Process …


In [10]:
# Extract fields required for text cleaning and EDA

text_df = df[["id", "title", "category", "description"]].copy()

# Extract the category label from the nested category field
text_df["category_name"] = text_df["category"].apply(
    lambda x: x.get("label") if isinstance(x, dict) else None
)

# Rename the job identifier for clarity
text_df.rename(columns={"id": "job_id"}, inplace=True)

# Keep only the fields needed for the text-data track
text_df = text_df[
    ["job_id", "title", "category_name", "description"]
]

print("Text dataset shape:", text_df.shape)
print("\nColumns:")
print(text_df.columns.tolist())

print("\nJob categories:")
print(text_df["category_name"].value_counts())

Text dataset shape: (2500, 4)

Columns:
['job_id', 'title', 'category_name', 'description']

Job categories:
category_name
IT Jobs                             1713
PR, Advertising & Marketing Jobs     494
Accounting & Finance Jobs             72
Sales Jobs                            70
Engineering Jobs                      43
Creative & Design Jobs                37
Consultancy Jobs                      16
HR & Recruitment Jobs                 12
Teaching Jobs                         10
Admin Jobs                             9
Energy, Oil & Gas Jobs                 7
Scientific & QA Jobs                   5
Logistics & Warehouse Jobs             3
Trade & Construction Jobs              2
Legal Jobs                             2
Graduate Jobs                          2
Manufacturing Jobs                     1
Healthcare & Nursing Jobs              1
Customer Services Jobs                 1
Name: count, dtype: int64


In [11]:
# Preserve the original job description before preprocessing

text_df["description_original"] = text_df["description"]

# Create a separate working column for text preprocessing
text_df["description_clean"] = text_df["description"]

print("Original description preserved:", 
      text_df["description_original"].notna().all())

print("Clean description column created:", 
      "description_clean" in text_df.columns)

print("\nCurrent shape:", text_df.shape)

Original description preserved: True
Clean description column created: True

Current shape: (2500, 6)


In [12]:
# Check for HTML tags in job descriptions

html_pattern = r"<[^>]+>"

html_count = text_df["description_clean"].str.contains(
    html_pattern,
    regex=True,
    na=False
).sum()

print("Descriptions containing HTML tags:", html_count)
print("Total descriptions:", len(text_df))
print("Percentage containing HTML: {:.2f}%".format(
    html_count / len(text_df) * 100
))

Descriptions containing HTML tags: 0
Total descriptions: 2500
Percentage containing HTML: 0.00%


In [13]:
# Normalize job description text
# Technical symbols are preserved because they may represent skills.

text_df["description_clean"] = (
    text_df["description_clean"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Text normalization completed.")

print("\nOriginal description:")
print(text_df["description_original"].iloc[0][:500])

print("\nCleaned description:")
print(text_df["description_clean"].iloc[0][:500])

Text normalization completed.

Original description:
This job is with ABB, an inclusive employer and a member of myGwork – the largest global platform for the LGBTQ business community. Please do not contact the recruiter directly. At ABB, we help industries run leaner and cleaner—and every person here makes that happen. You’ll be empowered to lead, supported to grow, and proud of the impact we create together. Join us and help run what runs the world. This Position reports to: Digital Solution Engineering Manager What we believe in ABB’s Process …

Cleaned description:
this job is with abb, an inclusive employer and a member of mygwork – the largest global platform for the lgbtq business community. please do not contact the recruiter directly. at abb, we help industries run leaner and cleaner—and every person here makes that happen. you’ll be empowered to lead, supported to grow, and proud of the impact we create together. join us and help run what runs the world. this position reports

In [14]:
# Validate the text normalization

print("Original and cleaned descriptions stored separately:",
      "description_original" in text_df.columns and
      "description_clean" in text_df.columns)

print("All cleaned descriptions are lowercase:",
      text_df["description_clean"].str.islower().all())

print("Missing cleaned descriptions:",
      text_df["description_clean"].isna().sum())

# Check whether multiple consecutive spaces remain
multiple_spaces = text_df["description_clean"].str.contains(
    r"\s{2,}", regex=True, na=False
).sum()

print("Descriptions with multiple consecutive spaces:", multiple_spaces)

Original and cleaned descriptions stored separately: True
All cleaned descriptions are lowercase: False
Missing cleaned descriptions: 0
Descriptions with multiple consecutive spaces: 0


In [15]:
# Check whether alphabetic characters in the cleaned text are lowercase

def check_lowercase(text):
    letters = [char for char in text if char.isalpha()]
    return all(char.islower() for char in letters)

lowercase_check = text_df["description_clean"].apply(check_lowercase)

print("Descriptions with all alphabetic characters lowercase:",
      lowercase_check.sum())

print("Descriptions not fully lowercase:",
      (~lowercase_check).sum())

Descriptions with all alphabetic characters lowercase: 2498
Descriptions not fully lowercase: 2


In [16]:
import nltk

nltk.download("stopwords")

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

print("Number of English stopwords:", len(stop_words))
print("Sample stopwords:", list(stop_words)[:20])

Number of English stopwords: 198
Sample stopwords: ['aren', "doesn't", "mustn't", 'which', 'off', 'too', 'doing', 'be', 'them', 'whom', 't', 'while', 'me', "hadn't", 'until', 'than', 'y', 'a', 'she', 'those']


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/mithresh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
# Inspect the effect of stopword removal on a sample description

sample_text = text_df["description_clean"].iloc[0]

sample_words = sample_text.split()

filtered_words = [
    word for word in sample_words
    if word not in stop_words
]

print("Original word count:", len(sample_words))
print("After stopword removal:", len(filtered_words))
print("Words removed:", len(sample_words) - len(filtered_words))

print("\nBefore:")
print(" ".join(sample_words[:50]))

print("\nAfter:")
print(" ".join(filtered_words[:50]))

Original word count: 85
After stopword removal: 53
Words removed: 32

Before:
this job is with abb, an inclusive employer and a member of mygwork – the largest global platform for the lgbtq business community. please do not contact the recruiter directly. at abb, we help industries run leaner and cleaner—and every person here makes that happen. you’ll be empowered to lead,

After:
job abb, inclusive employer member mygwork – largest global platform lgbtq business community. please contact recruiter directly. abb, help industries run leaner cleaner—and every person makes happen. you’ll empowered lead, supported grow, proud impact create together. join us help run runs world. position reports to: digital solution engineering manager believe


In [18]:
# Preserve important negation words during stopword removal

protected_words = {
    "no", "not", "nor", "never",
    "neither", "without"
}

custom_stop_words = stop_words - protected_words

print("Original stopwords:", len(stop_words))
print("Protected words:", sorted(protected_words))
print("Stopwords used for removal:", len(custom_stop_words))

Original stopwords: 198
Protected words: ['neither', 'never', 'no', 'nor', 'not', 'without']
Stopwords used for removal: 195


In [19]:
# Remove stopwords while preserving the normalized text column

def remove_stopwords(text):
    words = text.split()
    return " ".join(
        word for word in words
        if word not in custom_stop_words
    )

text_df["description_nostopwords"] = (
    text_df["description_clean"].apply(remove_stopwords)
)

print("Stopword removal completed.")

print("\nColumns:")
print(text_df.columns.tolist())

Stopword removal completed.

Columns:
['job_id', 'title', 'category_name', 'description', 'description_original', 'description_clean', 'description_nostopwords']


In [20]:
# Validate the stopword removal

sample_original = text_df["description_clean"].iloc[0]
sample_nostop = text_df["description_nostopwords"].iloc[0]

print("Original normalized text:")
print(sample_original[:500])

print("\nText after stopword removal:")
print(sample_nostop[:500])

print("\nProtected-word check:")
for word in sorted(protected_words):
    if word in sample_original.split():
        print(f"{word}:",
              "preserved" if word in sample_nostop.split() else "REMOVED")

Original normalized text:
this job is with abb, an inclusive employer and a member of mygwork – the largest global platform for the lgbtq business community. please do not contact the recruiter directly. at abb, we help industries run leaner and cleaner—and every person here makes that happen. you’ll be empowered to lead, supported to grow, and proud of the impact we create together. join us and help run what runs the world. this position reports to: digital solution engineering manager what we believe in abb’s process …

Text after stopword removal:
job abb, inclusive employer member mygwork – largest global platform lgbtq business community. please not contact recruiter directly. abb, help industries run leaner cleaner—and every person makes happen. you’ll empowered lead, supported grow, proud impact create together. join us help run runs world. position reports to: digital solution engineering manager believe abb’s process …

Protected-word check:
not: preserved


In [21]:
import spacy

print("spaCy version:", spacy.__version__)

spaCy version: 3.8.14


In [22]:
# Load the English spaCy model for lemmatization

nlp = spacy.load("en_core_web_sm")

print("English spaCy model loaded successfully.")

English spaCy model loaded successfully.


In [23]:
# Test lemmatization on one job description

sample_text = text_df["description_clean"].iloc[0]

doc = nlp(sample_text)

sample_lemmas = [
    token.lemma_
    for token in doc
    if not token.is_space
]

print("Original text:")
print(sample_text[:500])

print("\nLemmatized text:")
print(" ".join(sample_lemmas[:100]))

Original text:
this job is with abb, an inclusive employer and a member of mygwork – the largest global platform for the lgbtq business community. please do not contact the recruiter directly. at abb, we help industries run leaner and cleaner—and every person here makes that happen. you’ll be empowered to lead, supported to grow, and proud of the impact we create together. join us and help run what runs the world. this position reports to: digital solution engineering manager what we believe in abb’s process …

Lemmatized text:
this job be with abb , an inclusive employer and a member of mygwork – the large global platform for the lgbtq business community . please do not contact the recruiter directly . at abb , we help industry run lean and clean — and every person here make that happen . you ’ll be empower to lead , support to grow , and proud of the impact we create together . join we and help run what run the world . this position report to : digital solution engineering manager wh

In [24]:
# Lemmatize all job descriptions using spaCy

def lemmatize_text(text):
    doc = nlp(text)
    return " ".join(
        token.lemma_
        for token in doc
        if not token.is_space
    )

text_df["description_lemmatized"] = text_df["description_clean"].apply(
    lemmatize_text
)

print("Lemmatization completed.")

print("\nNew columns:")
print(text_df.columns.tolist())

print("\nSample lemmatized description:")
print(text_df["description_lemmatized"].iloc[0][:500])

Lemmatization completed.

New columns:
['job_id', 'title', 'category_name', 'description', 'description_original', 'description_clean', 'description_nostopwords', 'description_lemmatized']

Sample lemmatized description:
this job be with abb , an inclusive employer and a member of mygwork – the large global platform for the lgbtq business community . please do not contact the recruiter directly . at abb , we help industry run lean and clean — and every person here make that happen . you ’ll be empower to lead , support to grow , and proud of the impact we create together . join we and help run what run the world . this position report to : digital solution engineering manager what we believe in abb ’s process …
